# 使用 Hugging Face 微调多语言推理模型

作者：[Edward Beeching](https://huggingface.co/edbeeching)、[Quentin Gallouédec](https://huggingface.co/qgallouedec) 和 [Lewis Tunstall](https://huggingface.co/lewtun)

像 [OpenAI o3](https://openai.com/index/introducing-o3-and-o4-mini/) 这样的大型推理模型会生成思维链，以提高回答的准确性和质量。然而，这些模型大多使用英语进行推理，即使问题是用其他语言提出的。

在本 notebook 中，我们将展示如何微调 OpenAI 的开放权重推理模型 [OpenAI gpt-oss-20b](https://huggingface.co/openai/gpt-oss-20b)，使其能够有效地用多种语言进行推理。具体做法是：在模型的系统提示词中添加一个新的 _“推理语言”_ 选项，并使用 Hugging Face 的 [TRL 库](https://github.com/huggingface/trl)在一个多语言推理数据集上进行[监督微调](https://huggingface.co/learn/llm-course/chapter11/1)。

我们将介绍以下步骤：

1. **准备工作：** 安装所需的库。
2. **准备数据集：** 下载数据集并将其格式化以用于微调。
3. **准备模型：** 加载基础模型，并配置 [LoRA](https://huggingface.co/learn/llm-course/chapter11/4)（一种节省内存的技术）用于微调。
4. **微调：** 使用我们的多语言推理数据训练模型。
5. **推理：** 使用微调后的模型以不同语言生成推理响应。

最终得到的是一个多语言推理模型，能够用英语、西班牙语、法语、意大利语或德语生成思维链。你甚至可以 _混合使用多种语言_——例如，用西班牙语提问，要求用德语进行推理，并用西班牙语得到最终回答：

```txt
User:
    ¿Cuál es el capital de Australia?
Assistant reasoning:
    Okay, der Benutzer fragt nach der Hauptstadt Australiens. Ich erinnere mich, dass Canberra die Hauptstadt ist. Ich
    sollte das bestätigen. Lass mich sehen, ob es irgendwelche potenziellen Verwirrungen gibt. Der Benutzer könnte auch
    an der größten Stadt interessiert sein. Die größte Stadt ist Sydney, aber die Hauptstadt ist Canberra. Ich sollte
    das klarstellen. Vielleicht auch erwähnen, dass Canberra eine geplante Stadt ist und nicht die größte. Der Benutzer
    könnte auch nach der Geografie fragen. Vielleicht erwähne ich, dass Canberra im südwestlichen Teil der Australian
    Capital Territory liegt. Ich sollte die Antwort präzise und freundlich halten. Vielleicht auch erwähnen, dass
    Canberra oft mit Sydney verwechselt wird. Ich sollte sicherstellen, dass die Antwort klar und korrekt ist.
Assistant response:
    La capital de Australia es **Canberra**. Aunque es la ciudad más pequeña de las principales capitales del país, fue
    elegida en 1908 como la sede del gobierno federal para equilibrar la influencia entre las ciudades de Sydney y
    Melbourne. Canberra está ubicada en el Territorio de la Capital Australiana (ACT), en el este de Australia.
```

我们希望本教程能帮助使用代表性不足语言的 AI 开发者，提升 [`openai/gpt-oss-20b`](https://huggingface.co/openai/gpt-oss-20b) 在其母语中的可解释性。

> **注意：** 本 notebook 设计为在单张 80GB 显存的 H100 GPU 上运行。如果你使用的是显存更小的 GPU，可以在下面的超参数中减小批大小和序列长度。

## 准备工作

首先，我们来安装所有必需的库。先安装 PyTorch：

In [ ]:
%pip install torch --index-url https://download.pytorch.org/whl/cu128

接下来，安装其余依赖：

In [ ]:
%pip install "trl>=0.20.0" "peft>=0.17.0" "transformers>=4.55.0" trackio

最后，按如下方式登录你的 Hugging Face 账户：

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

现在我们已经安装了所需的库，接下来看看用于微调的数据集。

## 准备数据集

我们将使用 [Multilingual-Thinking](https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking)，这是一个推理数据集，其中的思维链已被翻译成法语、西班牙语和德语等多种语言。在该数据集上微调 `openai/gpt-oss-20b` 后，模型将学会用这些语言生成推理步骤，从而让使用这些语言的用户能够理解其推理过程。

<iframe
  src="https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking/embed/viewer/default/train"
  frameborder="0"
  width="100%"
  height="560px"
></iframe>



我们从 Hugging Face Hub 下载这个数据集：

In [ ]:
from datasets import load_dataset

dataset = load_dataset("HuggingFaceH4/Multilingual-Thinking", split="train")
dataset

这是一个只有 1,000 个样本的小型数据集，但对于像 `openai/gpt-oss-20b` 这样经过大量后训练的模型来说，这通常已经绰绰有余。我们来看其中一个训练样本：

In [ ]:
dataset[0]

`gpt-oss` 模型是基于 Harmony 响应格式训练的，该格式用于定义对话结构、生成推理输出以及组织函数调用。该格式旨在模仿 OpenAI Responses API，下表总结了数据集中使用的不同消息类型：

|||
| :---- | :--|
| `developer` | developer 消息用于为模型提供自定义指令（也就是我们通常所说的 `system` 角色）。 |
| `user` | user 消息用于向模型提供输入。 |
| `assistant` | 模型的输出，可以是工具调用，也可以是消息输出。输出还可能关联到某个特定的“channel”，用于标识该消息的意图。 |
| `analysis` | 模型用于其思维链的消息 |
| `final` | 标记为 final channel 的消息是打算展示给最终用户的消息，代表模型的回复。 |
| `messages` | 将上述内容组合成完整对话的消息列表。这是模型的输入。 |

如果你熟悉 [OpenAI 的消息格式](https://platform.openai.com/docs/api-reference/messages/object)，会发现它与之非常相似，但有一个重要区别：

> `assistant` 轮次包含两个特殊字段：`thinking` 字段包含模型的推理过程，`content` 字段包含给用户的最终回复。

为了微调模型，我们需要将这些消息转换为模型能够理解的格式。实践中，这是通过使用模型的 [_聊天模板_](https://huggingface.co/docs/transformers/chat_templating) 格式化每条消息，然后对生成的文本进行分词来完成的。TRL 库会自动完成这些操作，但我们还是一步步过一遍，以理解其工作原理。

为此，我们先加载分词器：

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("openai/gpt-oss-20b")

然后，我们可以使用分词器的 `apply_chat_template()` 方法来格式化消息：

In [ ]:
messages = dataset[0]["messages"]
conversation = tokenizer.apply_chat_template(messages, tokenize=False)
print(conversation)

这个聊天模板相当复杂，我们来仔细看看！首先，可以看到有特殊 token `<|start|>` 和 `<|end|>`，分别表示每条消息的开始和结束。还有一个 `<|return|>` token，用于标记对话的结束。这些 token 帮助模型理解对话的结构。

我们还可以看到有 _两种_ 系统消息： 

* 一种是默认的 `system` 消息，用于所有对话。在上面的示例中，它指的是文本 _"You are ChatGPT, a large language model trained by OpenAI..."_
* 另一种是特殊的 `developer` 消息，包含自定义指令（由 `messages` 对象中的 `system` 角色定义）。这让我们可以为模型提供额外的上下文，告诉它在特定对话中应当如何表现。在上面的示例中，它指的是文本 _"You are an AI chatbot with a lively and energetic personality."_

最后，可以看到助手的回复包含在一系列 _channel_ 中：

* `analysis` channel 用于模型的推理过程，模型可以在其中逐步思考用户的问题。在上面的示例中，它指的是法语文本 _"D'accord, l'utilisateur demande les tendances Twitter..."_
* `final` channel 用于模型给用户的最终回复。在上面的示例中，它指的是文本 _"Hey there!  While I can't check Twitter..."_

现在我们已经了解了数据集的准备方式，接下来为训练准备模型。

## 准备模型

为了准备训练用的模型，我们首先从 [Hugging Face Hub](https://huggingface.co) 下载权重。我们将使用 🤗 Transformers 中的 `AutoModelForCausalLM` 类来加载模型：

In [ ]:
import torch
from transformers import AutoModelForCausalLM, Mxfp4Config

quantization_config = Mxfp4Config(dequantize=True)
model_kwargs = dict(
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    use_cache=False,
    device_map="auto",
)

model = AutoModelForCausalLM.from_pretrained("openai/gpt-oss-20b", **model_kwargs)

这会以训练所需的配置加载模型。`attn_implementation` 设置为 `eager` 以获得更好的性能，`use_cache` 设置为 `False`，因为我们将使用梯度检查点来微调模型。

如果你熟悉 🤗 Transformers，可能会注意到我们使用了 `Mxfp4Config` 进行量化。这是专门针对 OpenAI 模型的配置，让我们能够使用一种名为 [MXFP4](https://en.wikipedia.org/wiki/Block_floating_point) 的特殊 4 位浮点格式进行混合精度训练，该格式针对 AI 工作负载进行了优化。

在训练模型之前，我们先生成一个示例回复，看看模型在默认设置下的表现。为此，我们需要对一个示例提示词进行分词，然后使用模型生成回复：

In [ ]:
messages = [
    {"role": "user", "content": "¿Cuál es el capital de Australia?"},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

output_ids = model.generate(input_ids, max_new_tokens=512)
response = tokenizer.batch_decode(output_ids)[0]
print(response)

在这个示例中，可以看到模型先用英语对问题进行推理，然后用西班牙语给出最终回复。这是模型的默认行为，我们来看看能否通过一些微调来改变它。 

为此，我们将使用一种名为 [LoRA](https://huggingface.co/learn/llm-course/chapter11/4)（低秩适配，Low-Rank Adaptation）的技术来微调模型。这种技术允许我们只调整模型中的少数特定层，对于像 `openai/gpt-oss-20b` 这样的大模型尤其有用。

首先，我们需要将模型包装为 `PeftModel`，并定义 LoRA 配置。我们将使用 [PEFT 库](https://github.com/huggingface/peft)中的 `LoraConfig` 类来完成：

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules="all-linear",
    target_parameters=[
        "7.mlp.experts.gate_up_proj",
        "7.mlp.experts.down_proj",
        "15.mlp.experts.gate_up_proj",
        "15.mlp.experts.down_proj",
        "23.mlp.experts.gate_up_proj",
        "23.mlp.experts.down_proj",
    ],
)
peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

这里我们为 LoRA 使用了一些基础的超参数，但你可以尝试不同的取值，看看它们如何影响模型的性能。例如，增大 `r` 会增加可训练参数的数量，这可能会得到更好的模型，但代价是需要更多显存和训练时间。

**注意：** `openai/gpt-oss-20b` 模型采用的是[混合专家（MoE）](https://huggingface.co/blog/moe)架构。除了针对注意力层（`target_modules="all-linear"`）之外，将专家模块中的投影层也纳入进来同样很重要。PEFT 通过 `target_parameters` 参数支持这一点，它允许你指定专家相关的层，例如 `mlp.experts.down_proj` 和 `mlp.experts.gate_up_proj`。在本示例中，我们只针对其中一部分投影层，但我们鼓励你尝试不同的配置。

现在模型和数据集都已准备就绪，我们可以定义训练的超参数了。

## 微调

TRL 提供了一种使用 `SFTConfig` 类定义训练超参数的便捷方式。我们将按如下方式设置学习率、批大小、训练轮数（epoch）等参数：

In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    learning_rate=2e-4,
    gradient_checkpointing=True,
    num_train_epochs=1,
    logging_steps=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    max_length=2048,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine_with_min_lr",
    lr_scheduler_kwargs={"min_lr_rate": 0.1},
    output_dir="gpt-oss-20b-multilingual-reasoner",
    report_to="trackio",
    push_to_hub=True,
)

注意，`per_device_train_batch_size` 设置为 4，`gradient_accumulation_steps` 也设置为 4。这意味着在 1 张 GPU 上的有效批大小为 4 x 4 = 16。你可能需要根据自己的硬件配置调整这些值。我们还使用 [Trackio](https://huggingface.co/blog/trackio) 记录训练进度和指标，不过你也可以使用任何其他自己喜欢的日志库。

现在我们已经具备了训练模型所需的全部组件。我们将使用 TRL 中的 `SFTTrainer` 类来处理训练过程。训练器会负责格式化数据集、应用聊天模板并训练模型：

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)
trainer.train()

在 H100 GPU 上，训练大约需要 18 分钟，但根据你的硬件，可能需要更长时间。

## 保存模型并推送到 Hugging Face Hub

最后，你可以将微调后的模型推送到你的 Hub 仓库，与社区分享：

In [ ]:
trainer.save_model(training_args.output_dir)
trainer.push_to_hub(dataset_name="HuggingFaceH4/Multilingual-Thinking")

**注意**：为避免显存不足（OOM）错误，我们建议此时重启内核。训练好的模型仍然占用着 GPU 显存，但已经不再需要了。

## 推理

模型上传到 Hub 后，就可以用它进行推理了。为此，我们首先初始化原始的基础模型及其分词器。接下来，需要将微调后的权重与基础模型合并，以实现快速推理：

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("openai/gpt-oss-20b")

# Load the original model first
model_kwargs = dict(attn_implementation="eager", torch_dtype="auto", use_cache=True, device_map="auto")
base_model = AutoModelForCausalLM.from_pretrained("openai/gpt-oss-20b", **model_kwargs).cuda()

# Merge fine-tuned weights with the base model
peft_model_id = "gpt-oss-20b-multilingual-reasoner"
model = PeftModel.from_pretrained(base_model, peft_model_id)
model = model.merge_and_unload()

模型加载完毕后，最后一步就是用它生成一些 token！这里我们使用模型的 `generate` 方法，根据输入提示词生成输出。我们先定义提示词：

现在，我们可以对提示词进行分词并生成输出。最后，对输出 token 进行解码，得到最终回复：

In [ ]:
REASONING_LANGUAGE = "German"
SYSTEM_PROMPT = f"reasoning language: {REASONING_LANGUAGE}"
USER_PROMPT = "¿Cuál es el capital de Australia?"  # Spanish for "What is the capital of Australia?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

gen_kwargs = {"max_new_tokens": 512, "do_sample": True, "temperature": 0.6, "top_p": None, "top_k": None}

output_ids = model.generate(input_ids, **gen_kwargs)
response = tokenizer.batch_decode(output_ids)[0]
print(response)

我们再试试模型没有专门微调过的语言，例如中文和印地语：

In [ ]:
REASONING_LANGUAGE = "Chinese"  # or Hindi, or any other language...
SYSTEM_PROMPT = f"reasoning language: {REASONING_LANGUAGE}"
USER_PROMPT = "What is the national symbol of Canada?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

output_ids = model.generate(input_ids, **gen_kwargs)
response = tokenizer.batch_decode(output_ids)[0]
print(response)

太好了，它成功了——我们已经将 `openai/gpt-oss-20b` 微调为能够用多种语言进行推理！

## 总结

恭喜！你已经成功使用 TRL 库和 LoRA 微调了一个多语言推理模型。本 notebook 中的步骤可以加以调整，用于在 Hugging Face Hub 上的许多其他[数据集](https://huggingface.co/datasets)上微调 [`openai/gpt-oss-20b`](https://huggingface.co/openai/gpt-oss-20b)——我们非常期待看到你的作品！